# Track 1: Expanded Dataset Training — LeWM-VC

Trains on the full PEViD-HD dataset (21 clips) instead of just 2.
Optionally add VIRAT Ground videos for more diversity.

**Colab usage limit:** 2-3 hours/day. Saves progress to Google Drive.
Resume across sessions.

**Session workflow:**
- **First run:** Cells 1-3 (setup) → Cell 4 (download PEViD-HD) → Cell 5 (Phase 1)
- **Next session:** Cells 1-3 (setup) → Cell 5 (resume Phase 1) or Cell 6 (Phase 2)
- **Final session:** Cells 1-3 (setup) → Cell 6 (resume Phase 2) → Cell 7 (validate)

In [1]:
# ── Mount Google Drive for persistent storage ──
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/lewm_vc_track1"
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/logs", exist_ok=True)
print(f"Project directory: {PROJECT_DIR}")

Mounted at /content/drive
Project directory: /content/drive/MyDrive/lewm_vc_track1


In [42]:
# ── Clone repo and install deps (run once per session) ──
import os, sys, shutil
REPO_DIR = "/content/lewm-vc"

# Ensure a fresh clone by removing existing directory if it exists
if os.path.exists(REPO_DIR):
    print(f"Removing existing repository at {REPO_DIR} for a fresh clone...")
    shutil.rmtree(REPO_DIR)

print(f"Cloning repository to {REPO_DIR}...")
!git clone https://github.com/MAHAMAIA/lewm-vc.git "{REPO_DIR}"
sys.path.insert(0, f"{REPO_DIR}/src")
!pip install -q torch torchvision opencv-python pillow tqdm matplotlib pandas
!pip install -q ultralytics  # for validation
print("Setup complete")

Removing existing repository at /content/lewm-vc for a fresh clone...
Cloning repository to /content/lewm-vc...
Cloning into '/content/lewm-vc'...
remote: Enumerating objects: 290, done.
remote: Counting objects: 100% (290/290), done.
remote: Compressing objects: 100% (181/181), done.
remote: Total 290 (delta 101), reused 287 (delta 98), pack-reused 0 (from 0)
Receiving objects: 100% (290/290), 227.64 KiB | 418.00 KiB/s, done.
Resolving deltas: 100% (101/101), done.
Setup complete


In [3]:
# ── Session state — tracks progress across cells ──
import json, os, shutil

STATE_FILE = f"{PROJECT_DIR}/training_state.json"

def load_state():
    if os.path.exists(STATE_FILE):
        with open(STATE_FILE) as f:
            return json.load(f)
    return {
        "phase": "setup",
        "pevid_downloaded": False,
        "pevid_preprocessed": False,
        "pretrain_complete": False,
        "joint_epochs_run": 0,
        "lambda_rd": 0.05,
        "current_checkpoint": None,
    }

def save_state(state):
    with open(STATE_FILE, "w") as f:
        json.dump(state, f, indent=2)

state = load_state()
print(f"State phase: {state['phase']}")
print(f"PEViD-HD downloaded: {state['pevid_downloaded']}")
print(f"PEViD-HD preprocessed: {state['pevid_preprocessed']}")
print(f"Pretrain complete: {state['pretrain_complete']}")
print(f"Joint epochs run: {state['joint_epochs_run']}")

State phase: setup
PEViD-HD downloaded: False
PEViD-HD preprocessed: False
Pretrain complete: False
Joint epochs run: 0


## Step 4: Download & Preprocess PEViD-HD

Downloads the full 21-clip PEViD-HD surveillance dataset (~825 MB via FTP).
Converts each clip to 256×256 PNG frames on Colab, then copies to Drive.
Raw MPG files are deleted after conversion to save space.

**Optional: add VIRAT Ground videos** for more diversity.
VIRAT is not required — PEViD-HD alone (21 clips) gives 10x the current training data.

Run once. Interrupt-safe — processed clips are tracked and skipped on re-run.

In [6]:
# ── Download + Preprocess PEViD-HD ──
# FTP download from EPFL. Unzip one clip at a time, convert to PNGs,
# copy to Drive, delete raw. Drive budget: ≤10 GB.

import os, shutil, subprocess, sys
# ftplib is needed for direct FTP access
from ftplib import FTP

MAX_FRAMES_PER_CLIP = 400  # 16 seconds × 25 fps
COL_TMP = "/content/pevid_tmp"
DRIVE_FRAMES = f"{PROJECT_DIR}/data/pevid_frames"
os.makedirs(COL_TMP, exist_ok=True)
os.makedirs(DRIVE_FRAMES, exist_ok=True)

FTP_HOST = "tremplin.epfl.ch"
FTP_USER_FULL = "datasets@mmspgdata.epfl.ch"
FTP_USER = FTP_USER_FULL.split('@')[0] # Extract 'datasets'
FTP_PASS = "ohsh9jah4T"

# Use COL_RAW_MPG to store individual MPG files downloaded from FTP
COL_RAW_MPG = f"{COL_TMP}/raw_mpg_files"

processed = set(state.get("pevid_clips_processed", []))

# ── Check Drive budget ──
def get_drive_gb():
    total = 0
    for dp, _, fn in os.walk(DRIVE_FRAMES):
        for f in fn:
            total += os.path.getsize(os.path.join(dp, f))
    return total / 1e9

print(f"Drive usage: {get_drive_gb():.1f} GB")

# ── Download Raw MPG files from FTP (one-time) ──
if not state["pevid_downloaded"]:
    print("Connecting to FTP to download PEViD-HD clips...")
    os.makedirs(COL_RAW_MPG, exist_ok=True) # Ensure raw_mpg_files directory exists

    try:
        with FTP(FTP_HOST) as ftp:
            ftp.login(user=FTP_USER_FULL, passwd=FTP_PASS)
            ftp.cwd("/PEViD/PEViD-HD")
            remote_files = ftp.nlst()
            mpg_remote_files = [f for f in remote_files if f.endswith('.mpg')]

            print(f"Found {len(mpg_remote_files)} MPG clips on FTP server.")
            if not mpg_remote_files:
                print("No MPG files found on FTP. Please check path or server contents.")
                sys.exit(1)

            for remote_filename in mpg_remote_files:
                local_filepath = os.path.join(COL_RAW_MPG, remote_filename)
                if not os.path.exists(local_filepath):
                    print(f"Downloading {remote_filename} to {COL_RAW_MPG}...")
                    with open(local_filepath, 'wb') as fp:
                        ftp.retrbinary(f"RETR {remote_filename}", fp.write)
                    print(f"Downloaded {remote_filename}.")
                else:
                    print(f"Skipping download of {remote_filename}, already exists locally in {COL_RAW_MPG}.")

        state["pevid_downloaded"] = True # Mark as downloaded after all raw files are fetched
        save_state(state)
        print("All raw MPG files downloaded to Colab temporary storage.")
    except Exception as e:
        print(f"FTP download failed: {e}")
        state["pevid_downloaded"] = False
        save_state(state)
        print("Fatal error during FTP download. Please check network or FTP details.")
        sys.exit(1)

elif state["pevid_downloaded"]:
    print("Raw MPG files already downloaded. Skipping FTP download.")

# ── Process clips ──
if not state["pevid_preprocessed"]:
    # Iterate through files directly in COL_RAW_MPG.
    if not os.path.exists(COL_RAW_MPG) or not os.listdir(COL_RAW_MPG):
        print("Error: Raw MPG files directory is empty or does not exist for processing.")
        print("This might happen if previous download failed or files were manually removed.")
        sys.exit(1)

    # Get local mpg files from COL_RAW_MPG
    local_mpg_files = sorted([f for f in os.listdir(COL_RAW_MPG) if f.endswith('.mpg')])
    total_expected_clips = len(local_mpg_files) # Store the initial count of clips to process
    print(f"Found {total_expected_clips} MPG clips in local raw directory for processing.")

    for idx, raw_filename in enumerate(local_mpg_files):
        clip_name = os.path.splitext(raw_filename)[0]
        if clip_name in processed:
            print(f"Skipping processed clip: {clip_name}")
            continue

        # Check Drive budget
        if get_drive_gb() >= 10:
            print(f"  [stop] Drive budget reached ({get_drive_gb():.1f} GB)")
            break

        print(f"\n[{idx+1}/{total_expected_clips}] Processing {clip_name}...")

        raw_local_filepath = os.path.join(COL_RAW_MPG, raw_filename)

        # Convert to PNG frames
        out_dir = f"{COL_TMP}/frames/{clip_name}"
        os.makedirs(out_dir, exist_ok=True)
        !ffmpeg -i "{raw_local_filepath}" -vf "scale=256:256,fps=25" \
            -frames:v {MAX_FRAMES_PER_CLIP} -q:v 1 \
            "{out_dir}/frame_%04d.png" -y 2>/dev/null

        # Delete raw MPG after conversion
        os.remove(raw_local_filepath)

        # Copy to Drive
        drive_out = f"{DRIVE_FRAMES}/{clip_name}"
        shutil.copytree(out_dir, drive_out, dirs_exist_ok=True)
        shutil.rmtree(out_dir) # Remove temporary frames on Colab

        n_frames = len(os.listdir(drive_out))
        print(f"  {n_frames} frames → Drive ({get_drive_gb():.1f} GB used)")

        # Update state after successful processing of a clip
        processed.add(clip_name)
        state["pevid_clips_processed"] = sorted(list(processed))
        save_state(state)

    # Final cleanup: remove the temporary directory for raw MPG files if all processed
    if len(processed) == total_expected_clips: # All clips found initially are now processed
        shutil.rmtree(COL_RAW_MPG, ignore_errors=True)
        state["pevid_preprocessed"] = True
        save_state(state)
        print(f"\nAll {len(processed)} clips processed.")
    else:
        print(f"\nProcessed {len(processed)} out of {total_expected_clips} clips. Some clips may remain in {COL_RAW_MPG} if interrupted or budget exceeded.")

else:
    print("PEViD-HD already preprocessed.")

print(f"Final Drive usage: {get_drive_gb():.1f} GB")
print(f"Clips processed: {len(state.get('pevid_clips_processed', []))}")

Drive usage: 0.0 GB
Connecting to FTP to download PEViD-HD clips...
Found 21 MPG clips on FTP server.
Downloaded droppingBag_day_indoor_1_1.mpg.
Downloaded droppingBag_day_indoor_1_2.mpg.
Downloaded droppingBag_day_indoor_2_1.mpg.
Downloaded droppingBag_night_indoor_1_1.mpg.
Downloaded fighting_day_indoor_1_1.mpg.
Downloaded fighting_day_indoor_1_2.mpg.
Downloaded fighting_day_indoor_4_1.mpg.
Downloaded fighting_day_indoor_4_2.mpg.
Downloaded fighting_day_outdoor_1_1.mpg.
Downloaded fighting_day_outdoor_3_2.mpg.
Downloaded stealing_day_indoor_1_1.mpg.
Downloaded stealing_day_indoor_3_1.mpg.
Downloaded stealing_day_indoor_3_2.mpg.
Downloaded stealing_night_outdoor_1_1.mpg.
Downloaded stealing_night_outdoor_1_2.mpg.
Downloaded walking_day_indoor_3_1.mpg.
Downloaded walking_day_indoor_3_2.mpg.
Downloaded walking_day_outdoor_1_1.mpg.
Downloaded walking_day_outdoor_4_1.mpg.
Downloaded walking_day_outdoor_5_2.mpg.
Downloaded walking_night_outdoor_3_1.mpg.
All raw MPG files downloaded to Cola

## Step 5: Phase 1 — Predictor Pre-training

20 epochs, ~2 hours on T4. Resume-safe — tracks progress per epoch.
Checkpoints saved to Google Drive.

In [32]:
# ── Phase 1: Predictor Pre-training (resume-safe) ──
import sys, os
sys.path.insert(0, "/content/lewm-vc/src")

if state["pretrain_complete"]:
    print("Phase 1 already complete, skipping.")
else:
    TRAIN_SCRIPT = f"{PROJECT_DIR}/track1_train.py"

    # Pull latest source (batched encoding, context_len=6 fix)
    !cd /content/lewm-vc && git pull
    # Copy fresh training script to Drive
    !cp /content/lewm-vc/scripts/training/track1_train.py "{TRAIN_SCRIPT}"

    DATA_DIR = f"{PROJECT_DIR}/data/pevid_frames"
    CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
    LOG_FILE = f"{PROJECT_DIR}/logs/phase1_log.txt"

    existing = [f for f in os.listdir(CKPT_DIR) if 'phase1' in f]
    start_epoch = 0
    resume_ckpt = ""
    if existing:
        import glob
        ckpts = sorted(glob.glob(f"{CKPT_DIR}/phase1_epoch_*.pt"))
        if ckpts:
            import re
            last = ckpts[-1]
            m = re.search(r'epoch_(\d+)', last)
            if m:
                start_epoch = int(m.group(1))
                resume_ckpt = last

    remaining = 20 - start_epoch
    if remaining <= 0:
        state["pretrain_complete"] = True
        save_state(state)
        print("Phase 1 already complete.")
    else:
        print(f"Resuming Phase 1 from epoch {start_epoch + 1}/20")
        checkpoint_arg = f"--checkpoint {resume_ckpt}" if resume_ckpt else ""

        !cd "{PROJECT_DIR}" && PYTHONPATH=/content/lewm-vc/src python3 "{TRAIN_SCRIPT}" \
            --data-dir "{DATA_DIR}" \
            --phase pretrain \
            --epochs {remaining} \
            --batch-size 6 \
            --lr 1e-3 \
            {checkpoint_arg} \
            --output "{CKPT_DIR}/phase1_final.pt" \
            --workers 2 2>&1 | tee -a "{LOG_FILE}"

        state["pretrain_complete"] = os.path.exists(f"{CKPT_DIR}/phase1_final.pt")
        state["current_checkpoint"] = f"{CKPT_DIR}/phase1_final.pt"
        save_state(state)
        print(f"Phase 1 complete: {state['pretrain_complete']}")

Applying correct predictor context input and loss fix...
Patches applied to track1_train.py.
Resuming Phase 1 from epoch 1/20
/content/lewm-vc/src/lewm_vc/predictor.py:68: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Track 1 Training — device: cuda
Phase: pretrain, epochs: 20, batch: 6
λ_RD: 0.05, lr: 0.001
Data: /content/drive/MyDrive/lewm_vc_track1/data/pevid_frames
Total parameters: 20.0M
Scanning /content/drive/MyDrive/lewm_vc_track1/data/pevid_frames for frame sequences...
  Found 504 GOP segments across 21 videos
Train: 479 samples, Val: 25 samples

Phase 1: Predictor pre-training (20 epochs)
  Learning rate: 0.001
  Checkpoint: 
  Output: /content/drive/MyDrive/lewm_vc_track1/checkpoints/phase1_final.pt
  [warn] No checkpoint at , using random init
Epoch 1/20: 100%|██████████| 80/80 [01:16<00:00,  1.04it/s]
  Loss: 39.286754

## Step 6: Phase 2 — Joint Fine-Tuning

80 epochs, ~8 hours total across 3-4 Colab sessions.
Resume-safe — tracks epochs on Drive.

**Usage:** Run this cell. At the 2.5 hour mark, stop the runtime.
Next session: re-run cells 1-3, run this cell — it resumes automatically.

In [64]:
import sys, os, shutil
sys.path.insert(0, "/content/lewm-vc/src")
import torch

# Update lambda_rd
state["lambda_rd"] = 0.001
save_state(state)

if not state["pretrain_complete"]:
    print("[error] Phase 1 must complete before Phase 2. Run Step 5 first.")
else:
    CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
    LOG_FILE = f"{PROJECT_DIR}/logs/phase2_log.txt"
    TRAIN_SCRIPT = f"{PROJECT_DIR}/track1_train.py"
    LAMBDA = state.get("lambda_rd", 0.05)

    import glob, re
    joint_ckpts = sorted(glob.glob(f"{CKPT_DIR}/joint_epoch_*.pt"))
    start_epoch = 0
    resume_ckpt = f"{CKPT_DIR}/phase1_final.pt"
    if joint_ckpts:
        last = joint_ckpts[-1]
        m = re.search(r'epoch_(\d+)', last)
        if m:
            start_epoch = int(m.group(1))
            resume_ckpt = last

    TOTAL_PHASE2_EPOCHS = 10
    remaining = TOTAL_PHASE2_EPOCHS - start_epoch

    if remaining <= 0:
        print("Phase 2 already complete for this configuration.")
    else:
        print(f"Phase 2: Resuming from epoch {start_epoch + 1}/{TOTAL_PHASE2_EPOCHS}")
        print(f"Remaining: {remaining} epochs")

        # Pull latest source from repo, copy to Drive
        !cd /content/lewm-vc && git pull --ff-only
        !cp /content/lewm-vc/scripts/training/track1_train.py "{TRAIN_SCRIPT}"

        # Copy data from Drive to local SSD for fast I/O (one-time, ~30s)
        DRIVE_DATA = f"{PROJECT_DIR}/data/pevid_frames"
        LOCAL_DATA = "/content/pevid_frames_local"
        if not os.path.exists(LOCAL_DATA):
            print("Copying frames from Drive to local SSD (30s)...", flush=True)
            !cp -rn "{DRIVE_DATA}" /content/pevid_frames_local
            print("Copy complete.", flush=True)
        DATA_DIR = LOCAL_DATA

        # Load pretrained autoencoder and merge into Phase 1 checkpoint
        AE_DRIVE = f"{PROJECT_DIR}/checkpoints/autoencoder_final.pt"
        if not os.path.exists(AE_DRIVE):
            print("Please upload autoencoder_final.pt to:", AE_DRIVE)
            from google.colab import files
            uploaded = files.upload()
            for name in uploaded:
                shutil.move(name, AE_DRIVE)
                print(f"Saved {name} to {AE_DRIVE}")

        print(f"Loading pretrained autoencoder from {AE_DRIVE}...")
        ae_state = torch.load(AE_DRIVE, map_location="cpu", weights_only=True)

        if "encoder" in ae_state and "decoder" in ae_state:
            ae_encoder, ae_decoder = ae_state["encoder"], ae_state["decoder"]
            print(f"  Nested checkpoint: encoder keys={len(ae_encoder)}, decoder keys={len(ae_decoder)}")
        else:
            print(f"  Flat checkpoint: {len(ae_state)} total keys")
            ae_encoder = {k.replace("encoder.", ""): v for k, v in ae_state.items() if k.startswith("encoder.")}
            ae_decoder = {k.replace("decoder.", ""): v for k, v in ae_state.items() if k.startswith("decoder.")}

        print(f"  Sample encoder keys: {list(ae_encoder.keys())[:4]}")
        print(f"  Sample decoder keys: {list(ae_decoder.keys())[:4]}")

        p1 = torch.load(resume_ckpt, map_location="cpu", weights_only=True)
        p1["encoder"] = ae_encoder
        p1["decoder"] = ae_decoder
        merged_path = f"{CKPT_DIR}/phase1_with_ae.pt"
        torch.save(p1, merged_path)
        resume_ckpt = merged_path
        print(f"Merged pretrained autoencoder -> {merged_path}")

        !cd "{PROJECT_DIR}" && PYTHONPATH=/content/lewm-vc/src python3 "{TRAIN_SCRIPT}" \
            --data-dir "{DATA_DIR}" \
            --phase joint \
            --epochs {remaining} \
            --batch-size 4 \
            --lr 5e-5 \
            --lambda-rd {LAMBDA} \
            --checkpoint "{resume_ckpt}" \
            --output "{CKPT_DIR}/joint_lambda_{LAMBDA}_final.pt" \
            --workers 2 2>&1 | tee -a "{LOG_FILE}"

        state["current_checkpoint"] = f"{CKPT_DIR}/joint_lambda_{LAMBDA}_final.pt"
        save_state(state)
        print(f"Phase 2 output: {state['current_checkpoint']}")


Phase 2: Resuming from epoch 1/10
Remaining: 10 epochs
Quantizer device fix already applied.
Entropy sum fix already applied.
/content/lewm-vc/src/lewm_vc/predictor.py:68: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
Track 1 Training — device: cuda
Phase: joint, epochs: 10, batch: 4
λ_RD: 0.0, lr: 5e-05
Data: /content/drive/MyDrive/lewm_vc_track1/data/pevid_frames
Total parameters: 20.0M
Scanning /content/drive/MyDrive/lewm_vc_track1/data/pevid_frames for frame sequences...
  Found 504 GOP segments across 21 videos
Train: 479 samples, Val: 25 samples
  Loaded Phase 1 weights from /content/drive/MyDrive/lewm_vc_track1/checkpoints/phase1_final.pt
Epoch 1/10: 100%|██████████| 120/120 [03:23<00:00,  1.70s/it]
  Loss: 0.0538  (λ=0.0)
Epoch 2/10: 100%|██████████| 120/120 [03:22<00:00,  1.68s/it]
  Loss: 0.0315  (λ=0.0)
Epoch 3/10: 100%|█

### Install YOLOv5n for validation

In [38]:
# Install ultralytics and download yolov5nu.pt model for semantic probe.
!pip install -q ultralytics
!python -c "from ultralytics import YOLO; YOLO('yolov5nu.pt')"

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## Step 7: Validate Checkpoint

Run LeWM-Eval on held-out PEViD-HD clips to measure accuracy.

In [65]:
# ── Validate on held-out clips ──
import sys, os, json, glob

CKPT_PATH = state.get("current_checkpoint")
if not CKPT_PATH or not os.path.exists(CKPT_PATH):
    print("No checkpoint found. Complete Phase 2 first.")
else:
    TEST_DIR = f"{PROJECT_DIR}/data/pevid_frames"
    all_clips = sorted(glob.glob(f"{TEST_DIR}/*"))
    test_clips = [c for c in all_clips if 'dropping' in c or 'stealing' in c][:3]
    test_clips = test_clips or all_clips[-3:]  # fallback

    print(f"Validating on {len(test_clips)} held-out clips")
    results = []

    # Clone the evaluation suite separately
    EVAL_DIR = "/content/lewm-eval"
    if not os.path.exists(EVAL_DIR):
        !git clone https://github.com/MAHAMAIA/lewm-eval.git "{EVAL_DIR}"
    sys.path.insert(0, EVAL_DIR)

    # Install yolo teacher (redundant, moved to dedicated cell 7ed719cb)
    # !pip install -q ultralytics
    # !python -c "from ultralytics import YOLO; YOLO('yolov5nu.pt')" 2>/dev/null

    # Path to the semantic probe script from the newly cloned lewm-eval repo
    # Corrected path to include the 'evaluation' subdirectory
    semantic_probe_script_path = os.path.join(EVAL_DIR, 'evaluation', 'semantic_probe.py')

    for clip in test_clips:
        name = os.path.basename(clip)
        print(f"\n--- {name} ---")
        # Reduced --max-frames to prevent CUDA out of memory error
        !python "{semantic_probe_script_path}" \
            --frames "{clip}" \
            --teacher yolov5nu.pt \
            --epochs 30 --patience 10 \
            --max-frames 50 \
            --output "{PROJECT_DIR}/logs/val_{name}.json"
        val_file = f"{PROJECT_DIR}/logs/val_{name}.json"
        if os.path.exists(val_file):
            with open(val_file) as f:
                r = json.load(f)
            acc = r.get('class_accuracy', 'N/A')
            results.append((name, acc))
            print(f"  Accuracy: {acc}")

    print("\n--- Summary ---")
    for name, acc in results:
        print(f"  {name}: {acc}")

Validating on 3 held-out clips

--- droppingBag_day_indoor_1_1 ---
LeWM-Eval v0.1 — device: cuda

  Loaded 50 frames from /content/drive/MyDrive/lewm_vc_track1/data/pevid_frames/droppingBag_day_indoor_1_1
  Loading teacher: yolov5nu.pt...
  Generating labels: 100% 50/50 [00:01<00:00, 37.07it/s]
    Epoch 5/30 — loss=4.6131, train_acc=0.275, val_acc=0.400
    Epoch 10/30 — loss=2.3128, train_acc=0.725, val_acc=0.400
    Early stopping at epoch 11 (val_acc=0.400, best=0.600)
  Restored best validation accuracy: 0.600

Results saved to /content/drive/MyDrive/lewm_vc_track1/logs/val_droppingBag_day_indoor_1_1.json
  Accuracy: 0.6

--- droppingBag_day_indoor_1_2 ---
LeWM-Eval v0.1 — device: cuda

  Loaded 50 frames from /content/drive/MyDrive/lewm_vc_track1/data/pevid_frames/droppingBag_day_indoor_1_2
  Loading teacher: yolov5nu.pt...
  Generating labels: 100% 50/50 [00:01<00:00, 29.13it/s]
    Epoch 5/30 — loss=4.6019, train_acc=1.000, val_acc=1.000
    Epoch 10/30 — loss=1.8770, train_acc

In [47]:
# Inspect the contents of the newly cloned 'lewm-eval' directory
!ls -l /content/lewm-eval/

total 4972
-rw-r--r-- 1 root root   25861 May 22 07:24 colab_probe.ipynb
drwxr-xr-x 2 root root    4096 May 22 07:24 docs
drwxr-xr-x 2 root root    4096 May 22 07:24 evaluation
-rw-r--r-- 1 root root 5022707 May 22 07:24 mixkit-busy-street-in-the-city-4000-hd-ready.mp4
-rw-r--r-- 1 root root    1410 May 22 07:24 README.md
drwxr-xr-x 2 root root    4096 May 22 07:24 scripts
-rw-r--r-- 1 root root   18917 May 22 07:24 track1_colab.ipynb


In [44]:
# Inspect the contents of the 'evaluation' directory to verify the presence of semantic_probe.py
!ls -l /content/lewm-vc/evaluation/

ls: cannot access '/content/lewm-vc/evaluation/': No such file or directory


In [45]:
# Inspect the contents of the root lewm-vc directory
!ls -l /content/lewm-vc/

total 72
drwxr-xr-x 4 root root  4096 May 22 07:21 archive
drwxr-xr-x 2 root root  4096 May 22 07:21 benchmark_milestone1
drwxr-xr-x 2 root root  4096 May 22 07:21 benchmark_milestone2
drwxr-xr-x 2 root root  4096 May 22 07:21 benchmark_milestone3
drwxr-xr-x 2 root root  4096 May 22 07:21 benchmark_milestone4
drwxr-xr-x 2 root root  4096 May 22 07:21 benchmark_milestone4b
drwxr-xr-x 2 root root  4096 May 22 07:21 configs
drwxr-xr-x 2 root root  4096 May 22 07:21 experiment
drwxr-xr-x 2 root root  4096 May 22 07:21 ffmpeg
-rw-r--r-- 1 root root  1610 May 22 07:21 pyproject.toml
-rw-r--r-- 1 root root  4342 May 22 07:21 README.md
drwxr-xr-x 3 root root  4096 May 22 07:21 scripts
drwxr-xr-x 4 root root  4096 May 22 07:21 src
drwxr-xr-x 2 root root  4096 May 22 07:21 tests
-rw-r--r-- 1 root root 10613 May 22 07:21 train_local.py


In [66]:
# ── Cell 8: BPP + PSNR on test frames ──
import sys, os, json, glob
sys.path.insert(0, "/content/lewm-vc/src")
import torch
import numpy as np
from PIL import Image
from pathlib import Path
CKPT_PATH = state.get("current_checkpoint")
if not CKPT_PATH or not os.path.exists(CKPT_PATH):
    print("No checkpoint found. Complete Phase 2 first.")
else:
    from lewm_vc.encoder import LeWMEncoder
    from lewm_vc.working_decoder import LeWMDecoder
    from lewm_vc.entropy import HyperpriorEntropy
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading checkpoint: {CKPT_PATH}")
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
    print(f"Checkpoint keys: {list(ckpt.keys())}") # Debug: print checkpoint keys
    encoder = LeWMEncoder(latent_dim=192, patch_size=16).to(device)
    decoder = LeWMDecoder(latent_dim=192).to(device)
    entropy = HyperpriorEntropy(latent_dim=192).to(device)
    encoder.load_state_dict(ckpt["encoder"], strict=False)
    decoder.load_state_dict(ckpt["decoder"], strict=False)
    try:
        entropy.load_state_dict(ckpt["entropy"], strict=True)
        print("Entropy model loaded with strict=True.")
    except RuntimeError as e:
        print(f"Error loading entropy model with strict=True: {e}")
        print("Attempting to load with strict=False (may mask issues).")
        entropy.load_state_dict(ckpt["entropy"], strict=False)
    encoder.eval(); decoder.eval(); entropy.eval()
    print("Models loaded.")
    n_entropy_params = sum(p.sum().item() for p in entropy.parameters() if p.requires_grad)
    print(f"Entropy model params (sum): {n_entropy_params:.4f}") # Debug: print sum of entropy parameters
    test_clips = sorted(glob.glob(f"{DRIVE_FRAMES}/*"))
    clip = test_clips[-1]
    frames = sorted(glob.glob(f"{clip}/*.png"))[:10]
    print(f"Evaluating on {len(frames)} frames from {Path(clip).name}")
    @torch.no_grad()
    def compute_psnr(recon, target):
        mse = torch.mean((recon - target) ** 2).item()
        return 20 * np.log10(1.0 / np.sqrt(mse + 1e-10))
    bpps, psnrs = [], []
    for i, fpath in enumerate(frames):
        img = Image.open(fpath).convert("RGB").resize((256, 256))
        img_t = torch.from_numpy(np.array(img).astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
        z = encoder(img_t)
        print(f"  Frame {i+1} - Encoder output (z) mean: {z.mean().item():.6f}, std: {z.std().item():.6f}")
        qz = torch.round(z / (2.0/255.0)) * (2.0/255.0)
        print(f"  Frame {i+1} - Quantized latents (qz) mean: {qz.mean().item():.6f}, std: {qz.std().item():.6f}")
        recon = decoder(qz, target_size=(256, 256))
        print(f"  Frame {i+1} - Recon stats: mean={recon.mean().item():.4f}, std={recon.std().item():.4f}, min={recon.min().item():.4f}, max={recon.max().item():.4f}")
        rate, _ = entropy(qz)
        print(f"  Frame {i+1} - Entropy rate (raw) shape: {rate.shape}, mean: {rate.mean().item():.6f}, max: {rate.max().item():.6f}, min: {rate.min().item():.6f}") # Debug: print rate tensor details
        total_bits = rate.sum().item()
        bpp = total_bits / (256 * 256 * 3)
        psnr = compute_psnr(recon.cpu(), img_t.cpu())
        bpps.append(bpp)
        psnrs.append(psnr)
    print(f"\n--- Results ---")
    print(f"Avg BPP:  {np.mean(bpps):.6f}")
    print(f"Avg PSNR: {np.mean(psnrs):.2f} dB")
    print()
    for i, (b, p) in enumerate(zip(bpps, psnrs)):
        print(f"  Frame {i+1}: BPP={b:.6f}  PSNR={p:.2f} dB")

Loading checkpoint: /content/drive/MyDrive/lewm_vc_track1/checkpoints/joint_lambda_0.0_final.pt
Checkpoint keys: ['encoder', 'decoder', 'predictor', 'entropy', 'lambda', 'epoch', 'loss']
Entropy model loaded with strict=True.
Models loaded.
Entropy model params (sum): 8.2960
Evaluating on 10 frames from walking_night_outdoor_3_1
  Frame 1 - Encoder output (z) mean: -0.021668, std: 0.483349
  Frame 1 - Quantized latents (qz) mean: -0.021678, std: 0.483358
  Frame 1 - Recon stats: mean=0.3558, std=0.1478, min=0.1925, max=1.0000
  Frame 1 - Entropy rate (raw) shape: torch.Size([1, 1, 16, 16]), mean: 29.553234, max: 29.654888, min: 29.501760
  Frame 2 - Encoder output (z) mean: -0.021640, std: 0.483341
  Frame 2 - Quantized latents (qz) mean: -0.021647, std: 0.483330
  Frame 2 - Recon stats: mean=0.3558, std=0.1478, min=0.1927, max=1.0000
  Frame 2 - Entropy rate (raw) shape: torch.Size([1, 1, 16, 16]), mean: 29.553196, max: 29.655094, min: 29.501453
  Frame 3 - Encoder output (z) mean: -0

In [ ]:
# ── Cell 9: Compare against x265 (RD curve) ──
import sys, os, json, subprocess, tempfile
sys.path.insert(0, "/content/lewm-vc/src")
import torch
import numpy as np
from PIL import Image
from pathlib import Path
CKPT_PATH = state.get("current_checkpoint")
if not CKPT_PATH or not os.path.exists(CKPT_PATH):
    print("No checkpoint found. Complete Phase 2 first.")
else:
    from lewm_vc.encoder import LeWMEncoder
    from lewm_vc.working_decoder import LeWMDecoder
    from lewm_vc.entropy import HyperpriorEntropy
    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=True)
    encoder = LeWMEncoder(latent_dim=192, patch_size=16).to(device)
    decoder = LeWMDecoder(latent_dim=192).to(device)
    entropy = HyperpriorEntropy(latent_dim=192).to(device)
    encoder.load_state_dict(ckpt["encoder"], strict=False)
    decoder.load_state_dict(ckpt["decoder"], strict=False)
    entropy.load_state_dict(ckpt["entropy"], strict=False)
    encoder.eval(); decoder.eval(); entropy.eval()
    # Load test frames
    test_clips = sorted(glob.glob(f"{DRIVE_FRAMES}/*"))
    clip = test_clips[-1]
    frames = sorted(glob.glob(f"{clip}/*.png"))[:16]
    print(f"Testing on {len(frames)} frames from {Path(clip).name}")
    def imgs_to_y4m(imgs, path, fps=25):
        """Write PNG frames to Y4M for x265."""
        from PIL import Image as PILImg
        h, w = 256, 256
        with open(path, 'wb') as f:
            f.write(f"YUV4MPEG2 W{w} H{h} F{fps}:1 Ip A0:0 C444\n".encode())
            for img in imgs:
                f.write("FRAME\n".encode())
                yuv = img.convert("YCbCr")
                f.write(bytes(yuv.tobytes()))
    def psnr_from_mse(mse):
        return 20 * np.log10(1.0 / np.sqrt(mse + 1e-10))
    # LeWM-VC eval
    print("\n--- LeWM-VC (your model) ---")
    lewm_bpps, lewm_psnrs = [], []
    for fpath in frames:
        img = Image.open(fpath).convert("RGB").resize((256, 256))
        img_t = torch.from_numpy(np.array(img).astype(np.float32) / 255.0).permute(2, 0, 1).unsqueeze(0).to(device)
        z = encoder(img_t)
        qz = torch.round(z / (2.0/255.0)) * (2.0/255.0)
        recon = decoder(qz, target_size=(256, 256))
        rate = entropy(qz).sum().item()
        bpp = rate / np.log(2) / (256 * 256 * 3)
        mse = torch.mean((recon.cpu() - img_t.cpu()) ** 2).item()
        lewm_bpps.append(bpp)
        lewm_psnrs.append(psnr_from_mse(mse))
    lewm_avg_bpp = np.mean(lewm_bpps)
    lewm_avg_psnr = np.mean(lewm_psnrs)
    print(f"  Avg BPP: {lewm_avg_bpp:.4f}  |  Avg PSNR: {lewm_avg_psnr:.2f} dB")
    # x265 eval at multiple CRF values
    print("\n--- x265 (H.265) ---")
    crf_values = [18, 22, 26, 30, 34, 38]
    x265_results = []
    tmp_dir = tempfile.mkdtemp()
    for frame_i in frames:
        shutil.copy(frame_i, f"{tmp_dir}/frame_{Path(frame_i).name}")
    for crf in crf_values:
        out_path = f"{tmp_dir}/x265_crf{crf}.mp4"
        subprocess.run(
            ["ffmpeg", "-y", "-framerate", "25", "-i", f"{tmp_dir}/frame_%04d.png",
             "-c:v", "libx265", "-crf", str(crf), "-pix_fmt", "yuv444p",
             "-preset", "medium", out_path],
            capture_output=True, check=True
        )
        # Decode back to frames and measure PSNR
        decoded_dir = f"{tmp_dir}/decoded_crf{crf}"
        os.makedirs(decoded_dir, exist_ok=True)
        subprocess.run(
            ["ffmpeg", "-y", "-i", out_path, f"{decoded_dir}/frame_%04d.png"],
            capture_output=True, check=True
        )
        # Compute bitrate from file size
        file_bits = os.path.getsize(out_path) * 8
        duration_s = len(frames) / 25.0
        bitrate_bps = file_bits / duration_s
        bpp = bitrate_bps / (256 * 256 * 25 * 3)  # bits per pixel per color
        # PSNR
        psnrs = []
        for i, fpath in enumerate(frames):
            orig = np.array(Image.open(fpath).resize((256, 256))).astype(np.float32) / 255.0
            dec = np.array(Image.open(f"{decoded_dir}/frame_{i+1:04d}.png").resize((256, 256))).astype(np.float32) / 255.0
            mse = np.mean((orig - dec) ** 2)
            psnrs.append(psnr_from_mse(mse))
        avg_psnr = np.mean(psnrs)
        x265_results.append((bpp, avg_psnr, crf))
        print(f"  CRF {crf}: BPP={bpp:.4f}  PSNR={avg_psnr:.2f} dB")
    shutil.rmtree(tmp_dir, ignore_errors=True)
    # Summary
    print("\n" + "="*55)
    print(f"{'Codec':<20} {'BPP':<10} {'PSNR (dB)':<12}")
    print("-"*55)
    print(f"{'LeWM-VC (yours)':<20} {lewm_avg_bpp:<10.4f} {lewm_avg_psnr:<12.2f}")
    for bpp, psnr, crf in x265_results:
        print(f"{'x265 CRF ' + str(crf):<20} {bpp:<10.4f} {psnr:<12.2f}")
    print("="*55)

## Download Checkpoint

Run this to download the trained checkpoint from Drive.

In [ ]:
# ── Download trained checkpoint ──
from google.colab import files
CKPT_PATH = state.get("current_checkpoint")
if CKPT_PATH and os.path.exists(CKPT_PATH):
    files.download(CKPT_PATH)
    print(f"Downloaded: {CKPT_PATH}")
else:
    print("No checkpoint to download.")